# Async CUDA Profiling with Nsight Systems
Profiling multiple streams and kernels using `nsys` in Colab.

In [ ]:
# 单元格1：安装 Nsight Systems（Colab 默认未安装）
!apt-get update -qq && apt-get install -y -qq nvidia-nsight-systems 2>&1 | tail -5

### 验证安装

In [ ]:
!nsys --version

### 编译

In [ ]:
!nvcc multi_stream.cu -o multi_stream
print("Compilation done.")

### 运行 nsys profile

In [ ]:
!nsys profile \
  --trace=cuda \
  --output=report \
  --force-overwrite \
  ./multi_stream 2>&1 | tail -10

### 导出 SQLite 数据库

In [ ]:
!nsys export report.qdrep --output=report.sqlite --type=sqlite 2>&1 | tail -5

### 查询 Profiling 结果

In [ ]:
import sqlite3
conn = sqlite3.connect('report.sqlite')
cursor = conn.cursor()

# 列出所有 kernel 及其平均耗时（微秒 -> 毫秒）
cursor.execute("""
    SELECT name,
           CAST(AVG(dur) AS REAL) / 1e6 AS avg_ms,
           COUNT(*) AS calls
    FROM CUPTI_ACTIVITY_KIND_KERNEL
    GROUP BY name
""")
rows = cursor.fetchall()
print(f"{'Kernel Name':<30} {'Avg Time (ms)':<20} {'Calls':<10}")
print("-"*60)
for row in rows:
    print(f"{row[0]:<30} {row[1]:<20.4f} {row[2]:<10}")

# 也可以查看 memcpy 信息（可选）
print("\n--- Memory Copies ---")
cursor.execute("""
    SELECT name,
           CAST(AVG(dur) AS REAL) / 1e6 AS avg_ms,
           COUNT(*) AS calls
    FROM CUPTI_ACTIVITY_KIND_MEMCPY
    GROUP BY name
""")
rows2 = cursor.fetchall()
for row in rows2:
    print(f"{row[0]:<40} {row[1]:<20.4f} {row[2]:<10}")

conn.close()